<a href="https://colab.research.google.com/github/Apichaya-bu/229352-stat-learning-for-data2/blob/main/%E0%B8%AA%E0%B8%B3%E0%B9%80%E0%B8%99%E0%B8%B2%E0%B8%82%E0%B8%AD%E0%B8%87_Lab07_Boosted_trees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Statistical Learning for Data Science 2 (229352)
#### Instructor: Donlapark Ponnoprat

#### [Course website](https://donlapark.pages.dev/229352/)

## Lab #6

## Boosted tree models on a simulated dataset

- [AdaBoostClassifier documentation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html#sklearn-ensemble-adaboostclassifier)
- [XGBClassifier documentation](https://xgboost.readthedocs.io/en/stable/python/python_api.html#xgboost.XGBClassifier)
- [LGBMClassifier documentation](https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMClassifier.html#lightgbm-lgbmclassifier)
- [GridSeachCV documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)


- [Data](https://github.com/donlapark/ds352-labs/raw/main/Lab06-data.zip)


Perform GridSearchCV of the following three models on the provided training set (`X_train.csv` and `y_train.csv`)

1. Evaluate these models on the test set (`X_test.csv` and `y_test.csv`). **Keep searching (using cross-validation) until you find the model that achieves > 0.83 out-of-fold accuracy (use `GridSeachCV.best_score_` to obtain the out-of-fold accuracy)**

2. Report the test accuracy of your best model.

3. For each model, plot the feature importances

For `AdaBoostClassifier`, feature importances can be obtained by calling the `feature_importances_` attribute after fitting the model.

For `XGBClassifier` and `LGBMClassifier`, feature importances can be obtained using the library’s `plot_importance` function. Here is a minimal example in XGBoost:

In [ ]:
from sklearn import datasets


iris = datasets.load_iris()
X = iris.data
y = iris.target

In [ ]:
from sklearn.ensemble import AdaBoostClassifier


ab = AdaBoostClassifier()
ab.fit(X, y)
ab.feature_importances_

In [ ]:
from xgboost import XGBClassifier, plot_importance


model = XGBClassifier()
model.fit(X, y)
plot_importance(model);

In [ ]:
X

In [ ]:
from xgboost import plot_tree

plot_tree(model, num_trees=4);

In [ ]:
import pandas as pd
import os

X_train = pd.read_csv('Lab06-data/X_train.csv', header=None)
y_train = pd.read_csv('Lab06-data/y_train.csv', header=None).values.ravel()
X_test = pd.read_csv('Lab06-data/X_test.csv', header=None)
y_test = pd.read_csv('Lab06-data/y_test.csv', header=None).values.ravel()


In [ ]:
import pandas as pd
import os

# Check if the data directory already exists to avoid re-downloading
if not os.path.exists('Lab06-data'):
    # Download the zip file
    !wget https://github.com/donlapark/ds352-labs/raw/main/Lab06-data.zip
    # Unzip the file
    !unzip Lab06-data.zip

# Now read the CSV files from the extracted directory
X_train = pd.read_csv('Lab06-data/X_train.csv', header=None)

X_train

--2026-01-19 05:50:32--  https://github.com/donlapark/ds352-labs/raw/main/Lab06-data.zip
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/donlapark/ds352-labs/main/Lab06-data.zip [following]
--2026-01-19 05:50:32--  https://raw.githubusercontent.com/donlapark/ds352-labs/main/Lab06-data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5611 (5.5K) [application/zip]
Saving to: ‘Lab06-data.zip.1’

Lab06-data.zip.1    100%[===================>]   5.48K  --.-KB/s    in 0s      

2026-01-19 05:50:32 (20.4 MB/s) - ‘Lab06-data.zip.1’ saved [5611/5611]

Archive:  Lab06-data.zip
replace X_test.csv? [y]es

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score

# 1. AdaBoost
param_ada = {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 1.0]}
grid_ada = GridSearchCV(AdaBoostClassifier(random_state=42), param_ada, cv=5)
grid_ada.fit(X_train, y_train)

# 2. XGBoost
param_xgb = {'n_estimators': [50, 100], 'max_depth': [3, 5, 7], 'learning_rate': [0.01, 0.1]}
grid_xgb = GridSearchCV(XGBClassifier(random_state=42), param_xgb, cv=5)
grid_xgb.fit(X_train, y_train)

# 3. LightGBM
param_lgbm = {'n_estimators': [50, 100], 'num_leaves': [31, 50], 'learning_rate': [0.01, 0.1]}
grid_lgbm = GridSearchCV(LGBMClassifier(random_state=42, verbose=-1), param_lgbm, cv=5)
grid_lgbm.fit(X_train, y_train)


In [ ]:
best_model = grid_xgb
print(f"Best Out-of-fold Accuracy: {best_model.best_score_:.4f}")

y_pred = best_model.predict(X_test)
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")

from xgboost import plot_importance
plot_importance(best_model.best_estimator_)
